In [1]:
%%capture
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import scipy as sp
from sklearn.preprocessing import LabelEncoder
from recsys_pipeliner.recommendations.transformer import (
    SimilarityTransformer,
    UserItemMatrixTransformer,
)
from recsys_pipeliner.algorithms.recommenders import ItemBasedCFRecommender
from IPython.display import display
from recsys_pipeliner.evaluation import (
    AccuracyMetrics,
    AlgorithmEvaluator,
    EvaluationDataset,
    TopNMetrics,
)
from IPython.display import display

In [3]:
# load test data
ratings_data_types = {"user_id": str, "item_id": str, "rating": np.float64}
user_item_ratings_df = pd.read_csv(
    "../../tests/test_data/user_item_ratings_toy.csv", dtype=ratings_data_types
)
item_features_df = pd.read_csv(
    "../../tests/test_data/item_features_toy.csv", index_col=0
)

display(user_item_ratings_df.head(3))
display(item_features_df.head(3))

,user_id,item_id,rating
0,U00001,I00024,0.8
1,U00001,I00013,0.6
2,U00001,I00005,1.0


,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10
I00001,0.65,0.71,0.75,0.91,0.70,0.75,0.88,0.95,0.62,0.87
I00002,0.57,0.56,0.14,0.33,0.39,0.29,0.17,0.13,0.11,0.60
I00003,0.46,0.65,0.51,0.69,0.47,0.49,0.49,0.59,0.68,0.72


In [4]:
# encode the user/item ids
item_encoder = LabelEncoder()
user_encoder = LabelEncoder()

user_item_ratings_df["item_id"] = item_encoder.fit_transform(user_item_ratings_df["item_id"])
user_item_ratings_df["user_id"] = user_encoder.fit_transform(user_item_ratings_df["user_id"])

user_item_ratings_np = user_item_ratings_df.to_numpy()

unique_users = pd.Series(user_encoder.classes_)
unique_items = pd.Series(item_encoder.classes_)

print("unique_users", unique_users.shape[0])
print("unique_items", unique_items.shape[0])

unique_users 12
unique_items 24


In [5]:
dataset = EvaluationDataset(
    user_item_ratings_np, 
    min_user_ratings=2, 
    min_item_ratings=2, 
    random_seed=42
)
trainset, testset = dataset.trainset, dataset.testset
anti_testset = dataset.anti_testset
usable_ratings = dataset.usable

print("user_item_ratings_np", user_item_ratings_np.shape)
print("trainset", trainset.shape)
print("testset", testset.shape)
print("anti_testset", anti_testset.shape)
print("usable_ratings", usable_ratings.shape)

assert set(np.unique(trainset[:, 0]).astype(int)) == set(unique_users.index)
assert (
    set(np.unique(testset[:, 0]).astype(int)) == set(unique_users.index)
)

user_item_ratings_np (96, 3)
trainset (84, 3)
testset (12, 3)
anti_testset (192, 2)
usable_ratings (96, 3)


In [6]:
# create the user/item matrix
user_item_matrix_transformer = UserItemMatrixTransformer()

user_item_matrix = user_item_matrix_transformer.transform(
    trainset,
)
print("user_item_matrix", user_item_matrix.shape)

# sanity check
users = trainset[:, 0].astype(int)
items = trainset[:, 1].astype(int)
ratings = trainset[:, 2].astype(np.float32)
for user, item, rating in zip(users, items, ratings):
    assert user_item_matrix[user, item] == rating

user_item_matrix (12, 24)


In [7]:
users = anti_testset[:, 0]
items = anti_testset[:, 1]

# sanity check
for user, item in anti_testset:
    assert user_item_matrix[user, item] == 0

In [8]:
rec = ItemBasedCFRecommender(k=5, n=5)
evaluator = AlgorithmEvaluator(rec)

result = evaluator.evaluate(
    user_item_matrix, dataset.testset, dataset.anti_testset, top_n=5
)

result

(AccuracyMetrics(rmse=0.3305, mae=0.2694),
 TopNMetrics(HR=0.25, cHR=0.25, ARHR=0.083333))

In [18]:
from sklearn.metrics.pairwise import cosine_similarity

# Content-based filtering and a hybrid recommender

item_features_np = item_features_df.to_numpy().astype(np.float32)
items = item_features_df.index.values
print("item_features_np", item_features_np.shape)
print("items", items)

item_similarity_matrix = cosine_similarity(item_features_np).astype(np.float32).round(6)
print("item_similarity_matrix", item_similarity_matrix.shape)

    

item_features_np (24, 10)
items ['I00001' 'I00002' 'I00003' 'I00004' 'I00005' 'I00006' 'I00007' 'I00008'
 'I00009' 'I00010' 'I00011' 'I00012' 'I00013' 'I00014' 'I00015' 'I00016'
 'I00017' 'I00018' 'I00019' 'I00020' 'I00021' 'I00022' 'I00023' 'I00024']
item_similarity_matrix (24, 24)
